In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
df = pd.read_csv('../../data/cleaned.csv')
df

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_45986/1552466201.py:1: DtypeWarning: Columns (0: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/cleaned.csv')


,ViewYN,PoolPrivateYN,ListingKey,CloseDate,ClosePrice,Latitude,Longitude,UnparsedAddress,LivingArea,DaysOnMarket,...,City_Wrightwood,City_Yermo,City_Yorba Linda,City_Yorkville,City_Yosemite,City_Yountville,City_Yreka,City_Yuba City,City_Yucaipa,City_Yucca Valley
0,0,0,1114722526,2025-05-23,1344000.0,37.893054,-122.086327,Unknown,-0.743210,-0.759852,...,False,False,False,False,False,False,False,False,False,False
1,1,1,1114680429,2025-05-05,2175000.0,34.129941,-118.426100,13438 Java Drive,0.196338,-0.441162,...,False,False,False,False,False,False,False,False,False,False
2,0,0,1114670553,2025-05-29,5975000.0,34.135406,-118.118478,1538 E California Boulevard,1.889271,-0.759852,...,False,False,False,False,False,False,False,False,False,False
3,0,0,1114650212,2025-05-28,1900000.0,34.037751,-118.395258,2807 Cardiff Avenue,-0.208957,-0.741106,...,False,False,False,False,False,False,False,False,False,False
4,0,0,1114573617,2025-05-27,640000.0,34.247108,-118.428730,13535 Goleta Street,-0.586134,0.383683,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136088,0,0,1065799115,2026-05-07,455000.0,33.786867,-118.187745,1584 Elm Avenue,-0.310766,12.775099,...,False,False,False,False,False,False,False,False,False,False
136089,1,0,1063517703,2026-05-08,1850000.0,34.377680,-119.145860,6770 Wheeler Canyon Road,0.926452,13.618690,...,False,False,False,False,False,False,False,False,False,False
136090,1,0,1052367803,2026-05-06,655000.0,34.275262,-118.452369,14655 Maclay Street,-1.005004,4.339188,...,False,False,False,False,False,False,False,False,False,False
136091,0,0,1048618285,2026-05-15,21500000.0,37.854648,-121.961587,7 Country Oak Ln,20.623036,14.799717,...,False,False,False,False,False,False,False,False,False,False


In [5]:
cat_col = ['UnparsedAddress', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName',  
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'PostalCode',  'ListingKey', 
         'ListingKeyNumeric', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate' ]

In [6]:
df_model = df.drop(columns=cat_col)

In [7]:
# Convert CloseDate to datetime
df_model['CloseDate'] = pd.to_datetime(df_model['CloseDate'])
df_model = df_model.sort_values('CloseDate').reset_index(drop=True)

# Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set

# Create testing and training sets, removing target and 'CloseDate'
train_df, test_df = get_time_split(df_model, X_months=6)
target = 'ClosePrice'
X_train = train_df.drop(columns=['ClosePrice', 'CloseDate'])
y_train = train_df[target]
X_test = test_df.drop(columns=['ClosePrice', 'CloseDate'])
y_test = test_df[target]

# When modeling, different testing windows will be evaulated
# window_options = [3, 6, 12]
# for X in window_options:
#     train_df, test_df = get_time_split(df, X_months=X)
    
#     print(f"--- Experimenting with X = {X} Months ---")
#     print(f"Train Date Range: {train_df['CloseDate'].min().strftime('%Y-%m-%d')} to {train_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(train_df)} rows)")
#     print(f"Test Date Range:  {test_df['CloseDate'].min().strftime('%Y-%m-%d')} to {test_df['CloseDate'].max().strftime('%Y-%m-%d')} ({len(test_df)} rows)\n")
    

In [8]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

In [10]:
from sklearn.metrics import r2_score

score = r2_score(y_test, y_pred)
print(score)

0.2843369857968536
